# TFM — Evaluación de modelos de pronóstico para FP&A
## Notebook orquestador

Importa los módulos de `src/` y ejecuta el experimento comparativo completo.
Cada familia de modelos está en su propio script; un `try/except` por bloque
permite omitir cualquier modelo que no esté disponible sin interrumpir el experimento.

In [5]:
import sys, os
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from collections import defaultdict
from tqdm.auto import tqdm

from src.config import CONFIG, SEED, TABLES_DIR, FIGURES_DIR
from src.utils import set_global_seed, get_logger
from src.backtesting import generate_folds, split_series, verify_no_leakage
from src.evaluation import compute_all, wape, error_contribution

set_global_seed(SEED)
logger = get_logger('main')
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

METRICS = ['MAE', 'RMSE', 'MAPE', 'sMAPE', 'MASE', 'WAPE']
MODEL_ORDER = ['SNaive', 'ETS', 'HoltWinters', 'SARIMA', 'LightGBM', 'MLP', 'NBEATS']

print(f'Config: horizon={CONFIG.horizon} | initial_train={CONFIG.initial_train} | step={CONFIG.step} | n_series={CONFIG.n_series}')
print(f'SEED={SEED}')


Config: horizon=18 | initial_train=54 | step=12 | n_series=200
SEED=42


## 1. Carga y preprocesamiento del dataset M4

In [6]:
from src.data_loader import load_m4_finance
from src.preprocessor import (
    filter_by_min_length, stratified_subsample,
    extract_series_list, get_volume_deciles
)

info_df, data_df = load_m4_finance(CONFIG)
info_filt, data_filt, lengths_filt = filter_by_min_length(info_df, data_df, CONFIG)
print('\nEstadísticas de longitud (series filtradas):')
print(lengths_filt.describe().round(1))

info_sub, data_sub = stratified_subsample(info_filt, data_filt, lengths_filt, CONFIG, seed=SEED)
series_list = extract_series_list(data_sub)
volume_deciles = get_volume_deciles(series_list)

print(f'\nSubmuestra: {len(series_list)} series | deciles asignados')


02:09:11 | INFO    | Descargando https://raw.githubusercontent.com/Mcompetitions/M4-methods/master/Dataset/M4-info.csv ...
INFO:data_loader:Descargando https://raw.githubusercontent.com/Mcompetitions/M4-methods/master/Dataset/M4-info.csv ...
02:09:12 | INFO    | Guardado en /content/tfm_FP-A/data/raw/M4-info.csv
INFO:data_loader:Guardado en /content/tfm_FP-A/data/raw/M4-info.csv
02:09:12 | INFO    | Descargando https://raw.githubusercontent.com/Mcompetitions/M4-methods/master/Dataset/Train/Monthly-train.csv ...
INFO:data_loader:Descargando https://raw.githubusercontent.com/Mcompetitions/M4-methods/master/Dataset/Train/Monthly-train.csv ...
02:10:35 | INFO    | Guardado en /content/tfm_FP-A/data/raw/Monthly-train.csv
INFO:data_loader:Guardado en /content/tfm_FP-A/data/raw/Monthly-train.csv
02:10:35 | INFO    | Cargando metadatos ...
INFO:data_loader:Cargando metadatos ...
02:10:35 | INFO    | Series Finance-Monthly encontradas: 10987
INFO:data_loader:Series Finance-Monthly encontradas: 


Estadísticas de longitud (series filtradas):
count    8493.0
mean      218.2
std       110.8
min        72.0
25%       155.0
50%       206.0
75%       269.0
max      1502.0
dtype: float64

Submuestra: 200 series | deciles asignados


In [7]:
# --- Exportar metadatos de las 200 series (id, cuartil, longitud, folds, decil) ---
max_folds = 5  # mismo tope usado en el resto del experimento

quartiles_full = pd.qcut(lengths_filt, q=4, labels=False, duplicates="drop")
n_folds_retenidos = [min(len(generate_folds(len(s), CONFIG)), max_folds) for s in series_list]

series_metadata = pd.DataFrame({
    "id": info_sub.index,
    "cuartil_longitud": quartiles_full.loc[info_sub.index].values,
    "longitud": info_sub["Actual_Length"].values,
    "n_folds": n_folds_retenidos,
    "decil_volumen": volume_deciles,
})
series_metadata.to_csv(TABLES_DIR / "series_metadata.csv", index=False)
print(f"Metadatos exportados: {TABLES_DIR / 'series_metadata.csv'} ({len(series_metadata)} series)")

Metadatos exportados: /content/tfm_FP-A/results/tables/series_metadata.csv (200 series)


## 2. Verificación del protocolo walk-forward

In [ ]:
import random
folds_per_series = [generate_folds(len(s), CONFIG) for s in series_list]
n_folds_list = [len(f) for f in folds_per_series]

print(f'Folds: min={min(n_folds_list)} | media={np.mean(n_folds_list):.1f} | max={max(n_folds_list)}')
print(f'Total evaluaciones: {sum(n_folds_list):,}')

random.seed(SEED)
sample_idx = random.sample(range(len(series_list)), min(20, len(series_list)))
for i in sample_idx:
    verify_no_leakage(generate_folds(len(series_list[i]), CONFIG), len(series_list[i]))
print('Verificacion leakage: OK')


In [ ]:
from src.visualization import plot_walk_forward_schema
fig_wf = plot_walk_forward_schema(
    series_length=150, initial_train=CONFIG.initial_train,
    horizon=CONFIG.horizon, step=CONFIG.step,
    n_folds=min(4, int(np.median(n_folds_list))),
    filename='fig01_walk_forward_schema.png'
)
print(f'Figura guardada: {fig_wf}')
from IPython.display import Image
Image(str(fig_wf))


## 3. Catálogo de modelos
Cada bloque `try/except` es independiente: si una librería falta o el modelo falla, se omite.

In [ ]:
LOCAL_MODELS = {}
GLOBAL_MODELS = {}

# Baseline — siempre disponible
from src.models.baseline import SeasonalNaive
LOCAL_MODELS['SNaive'] = SeasonalNaive
print('OK  SNaive')

try:
    from src.models.statistical import ETSForecaster
    LOCAL_MODELS['ETS'] = ETSForecaster
    print('OK  ETS')
except Exception as e:
    print(f'SKIP ETS: {e}')

try:
    from src.models.statistical import HoltWintersForecaster
    LOCAL_MODELS['HoltWinters'] = HoltWintersForecaster
    print('OK  HoltWinters')
except Exception as e:
    print(f'SKIP HoltWinters: {e}')

try:
    from src.models.statistical import SARIMAForecaster
    LOCAL_MODELS['SARIMA'] = SARIMAForecaster
    print('OK  SARIMA')
except Exception as e:
    print(f'SKIP SARIMA: {e}')

try:
    from src.models.ml import LightGBMForecaster
    GLOBAL_MODELS['LightGBM'] = LightGBMForecaster
    print('OK  LightGBM')
except Exception as e:
    print(f'SKIP LightGBM: {e}')

try:
    from src.models.ml import MLPForecaster
    GLOBAL_MODELS['MLP'] = MLPForecaster
    print('OK  MLP')
except Exception as e:
    print(f'SKIP MLP: {e}')

try:
    from src.models.ml import NBEATSForecaster
    GLOBAL_MODELS['NBEATS'] = NBEATSForecaster
    print('OK  N-BEATS')
except Exception as e:
    print(f'SKIP N-BEATS: {e}')

print(f'\nModelos locales : {list(LOCAL_MODELS.keys())}')
print(f'Modelos globales: {list(GLOBAL_MODELS.keys())}')


## 4. Backtesting walk-forward
### 4a. Modelos locales

In [ ]:
records_local = []

for model_name, ModelClass in LOCAL_MODELS.items():
    logger.info('Local: %s', model_name)
    for i, (series, decile) in enumerate(tqdm(
        zip(series_list, volume_deciles), total=len(series_list),
        desc=model_name, leave=False
    )):
        for fold in generate_folds(len(series), CONFIG):
            train, test = split_series(series, fold)
            try:
                m = ModelClass()
                m.fit(train)
                pred = m.predict(CONFIG.horizon)
                mets = compute_all(test, pred, train, CONFIG.seasonality)
                rec = {'model': model_name, 'series_idx': i, 'fold_id': fold.fold_id, 'decile': int(decile)}
                rec.update(mets)
            except Exception:
                rec = {'model': model_name, 'series_idx': i, 'fold_id': fold.fold_id, 'decile': int(decile),
                       **{m: np.nan for m in METRICS}}
            records_local.append(rec)

df_local = pd.DataFrame(records_local)
print(f'Registros: {len(df_local):,}')
print(df_local.groupby('model')['sMAPE'].mean().round(3).to_string())


### 4b. Modelos globales (un entrenamiento global por fold)

In [ ]:
records_global = []

if GLOBAL_MODELS:
    max_folds = max(len(generate_folds(len(s), CONFIG)) for s in series_list)
    for model_name, ModelClass in GLOBAL_MODELS.items():
        logger.info('Global: %s', model_name)
        for fold_id in tqdm(range(max_folds), desc=model_name, leave=False):
            trains, metas = [], []
            for i, (series, decile) in enumerate(zip(series_list, volume_deciles)):
                folds = generate_folds(len(series), CONFIG)
                if fold_id < len(folds):
                    fold = folds[fold_id]
                    train, test = split_series(series, fold)
                    trains.append(train)
                    metas.append((i, fold, test, series, int(decile)))
            if not trains:
                continue
            try:
                gm = ModelClass()
                gm.fit_global(trains)
                for i, fold, test, series, decile in metas:
                    train, _ = split_series(series, fold)
                    try:
                        pred = gm.predict(train, CONFIG.horizon)
                        mets = compute_all(test, pred, train, CONFIG.seasonality)
                        rec = {'model': model_name, 'series_idx': i, 'fold_id': fold_id, 'decile': decile}
                        rec.update(mets)
                    except Exception:
                        rec = {'model': model_name, 'series_idx': i, 'fold_id': fold_id, 'decile': decile,
                               **{m: np.nan for m in METRICS}}
                    records_global.append(rec)
            except Exception as ex:
                logger.warning('Fold %d fallo para %s: %s', fold_id, model_name, ex)

df_global = pd.DataFrame(records_global) if records_global else pd.DataFrame()
if not df_global.empty:
    print(f'Registros globales: {len(df_global):,}')
    print(df_global.groupby('model')['sMAPE'].mean().round(3).to_string())
else:
    print('Sin modelos globales ejecutados.')


## 5. Tabla comparativa de resultados

In [ ]:
frames = [df for df in [df_local, df_global] if df is not None and not df.empty]
df_all = pd.concat(frames, ignore_index=True)

present_order = [m for m in MODEL_ORDER if m in df_all['model'].unique()]

summary_rows = []
numeric_rows = []
for model in present_order:
    sub = df_all[df_all['model'] == model]
    row_str = {'Modelo': model}
    row_num = {'model': model}
    for metric in METRICS:
        vals = sub[metric].dropna()
        row_str[metric] = f'{vals.mean():.3f} +- {vals.std():.3f}' if len(vals) > 0 else 'N/A'
        row_num[metric + '_mean'] = vals.mean() if len(vals) > 0 else float('nan')
        row_num[metric + '_std']  = vals.std()  if len(vals) > 0 else float('nan')
    summary_rows.append(row_str)
    numeric_rows.append(row_num)

summary_table = pd.DataFrame(summary_rows).set_index('Modelo')
numeric_summary = pd.DataFrame(numeric_rows)

numeric_summary.to_csv(TABLES_DIR / 'resultados_comparativa.csv', index=False)
df_all.to_csv(TABLES_DIR / 'resultados_detalle.csv', index=False)

print('\n=== TABLA COMPARATIVA (media +- desviacion estandar) ===\n')
print(summary_table.to_string())
print(f'\nCSV guardados en: {TABLES_DIR}')


## 6. Visualizaciones

In [ ]:
from src.visualization import (
    plot_results_bar, plot_metrics_heatmap, plot_stability,
    plot_decile_contribution, plot_example_forecast
)

# Renombrar columnas para heatmap
hm_df = numeric_summary.rename(columns={m + '_mean': m for m in METRICS})

plot_results_bar(df_all, metric='sMAPE', filename='fig02_smape_comparativa.png')
plot_metrics_heatmap(hm_df, metrics=METRICS, filename='fig03_metricas_heatmap.png')

if 'fold_id' in df_local.columns and not df_local.empty:
    plot_stability(df_local, metric='sMAPE', filename='fig04_estabilidad_temporal.png')

dec_rows = []
for model in present_order:
    sub = df_all[df_all['model'] == model]
    for decile in sorted(sub['decile'].dropna().unique()):
        dsub = sub[sub['decile'] == decile]
        dec_rows.append({'model': model, 'decile': int(decile), 'abs_error': dsub['MAE'].dropna().sum()})
dec_df = pd.DataFrame(dec_rows)
total_err = dec_df.groupby('model')['abs_error'].transform('sum')
dec_df['contribution'] = dec_df['abs_error'] / total_err * 100
plot_decile_contribution(dec_df, filename='fig05_contribucion_deciles.png')

try:
    ex_series = series_list[0]
    fold_ex = generate_folds(len(ex_series), CONFIG)[-1]
    train_ex, test_ex = split_series(ex_series, fold_ex)
    preds_ex = {}
    for mn, MC in LOCAL_MODELS.items():
        try:
            m = MC(); m.fit(train_ex); preds_ex[mn] = m.predict(CONFIG.horizon)
        except: pass
    if preds_ex:
        plot_example_forecast(train_ex, test_ex, preds_ex, series_id='#0', filename='fig06_ejemplo_pronostico.png')
except Exception as e:
    print(f'Figura 6 omitida: {e}')

print(f'Figuras en: {FIGURES_DIR}')


## 7. Criterios de éxito

In [ ]:
SNAVE_REF = 14.45

print(f'{'Modelo':<15} {'sMAPE':>9} {'Delta(pp)':>10} {'MASE':>8} {'CV%':>7} {'Supera?':>9}')
print('-' * 62)
for model in present_order:
    sub = df_all[df_all['model'] == model]
    s_mean = sub['sMAPE'].dropna().mean()
    s_std  = sub['sMAPE'].dropna().std()
    mase   = sub['MASE'].dropna().mean()
    diff   = SNAVE_REF - s_mean
    cv     = s_std / s_mean * 100 if s_mean > 0 else float('nan')
    supera = 'SI' if diff >= 1.0 else ('marg.' if diff > 0 else 'NO')
    print(f'{model:<15} {s_mean:>9.3f} {diff:>+10.3f} {mase:>8.3f} {cv:>7.1f} {supera:>9}')

print(f'\nReferencia SNaive: {SNAVE_REF}%')
print('Supera baseline si reduccion >= 1 pp | Estable si CV < 25%')
